# P-Wave Detection — Google Colab

Classify **10-second** STEAD windows as **Noise** vs **Earthquake** with a 1D CNN.

**Runtime tip:** `Runtime → Change runtime type → GPU` (optional; CPU works).

## 1) Clone the repo

In [ ]:
# If the repo is private, use a GitHub PAT:
# !git clone https://<TOKEN>@github.com/mziroudi/P-Wave-Detection.git
!git clone https://github.com/mziroudi/P-Wave-Detection.git
%cd P-Wave-Detection
!git checkout cursor/p-wave-detection-5a2b || true
!ls

## 2) Install dependencies

In [ ]:
!pip install -q numpy pandas h5py matplotlib seaborn obspy scikit-learn tqdm requests
# Colab already has torch; reinstall only if needed:
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3) Download STEAD subsample (~800 MB test split)

In [ ]:
!python scripts/download_stead.py --test-only
!ls -lh data/stead_subsample/

## 4) Visualize waveforms (ObsPy)

In [ ]:
!python scripts/visualize_waveforms.py

from IPython.display import Image, display
display(Image('artifacts/waveforms/window_comparison.png'))
display(Image('artifacts/waveforms/earthquake_00.png'))

## 5) Prepare 10-second windows

In [ ]:
!python scripts/prepare_windows.py --prefer-split test --max-earthquake 3000 --max-noise 3000 --max-per-class 4000

## 6) Train the 1D CNN

In [ ]:
!python scripts/train.py --epochs 12 --batch-size 64

## 7) Evaluate + predict

In [ ]:
!python scripts/evaluate.py
!python scripts/predict.py --index 0
!python scripts/predict.py --index 1

from IPython.display import Image, display
display(Image('artifacts/confusion_matrix.png'))
display(Image('artifacts/roc_curve.png'))
display(Image('artifacts/prediction_samples.png'))

## Optional: skip training and use the shipped checkpoint

If you only want inference after preparing windows:

In [ ]:
# !python scripts/evaluate.py --checkpoint models/seismic_cnn1d_best.pt
# !python scripts/predict.py --checkpoint models/seismic_cnn1d_best.pt --index 0